In [1]:
from utils.imports import *

### 2.1.c)

This toy experiment yields results that could be easily perceived as hard proof and based upon which one can make the following case: A convolutional neural network is nothing else but a far smarter and well-constrained solution derived from the same basis as the one fully connected layers draw upon. The point of difference between FCN and a 1D-convolutional layer is entirely made from a dimensional mismatch at an output level: the output layer holds 10 neurons and activation maps (2, 5). In spite of this, the content remains exactly the same.

In [237]:
# 1. Declaring the variables (e.g., upper and lower bounds of the random initialization and number of elements equivalent to the number of input neurons);
elements:int = 5
lwb, upb = 0, 10
random_vector= torch.tensor([rdm.randint(lwb,upb) for item in range(elements)], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 2.1 Building the 1DCN and Linearity;
cnv1 = nn.Conv1d(in_channels=1, out_channels=2, padding='same', bias=True, kernel_size=3)
linear1 = nn.Linear(in_features=5, out_features=10, bias=True)

# 2.2 Weight transfer;
# 2.2.1 Zero out all the randomly initialized weights within W matrix of linear1;
linear1.weight.data.zero_()

# 3. Weight transfer loop mechanism
with torch.no_grad():
    linear1.weight.data.zero_()
    linear1.bias.data.zero_()
    all_Kernels = cnv1.weight.data
    pad:int = 1
    # 2. Loop through the 10 output rows
    for record in range(10):
        # 3. Determine which filter to use and calculate the local step 's'
        if record >= 5:
            current_kernel = all_Kernels[1, 0] # Grab filter 1
            s = record - 5                     # s resets to 0-4 for the second half
            linear1.bias.data[record:] = cnv1.bias.data[1]
        else:
            current_kernel = all_Kernels[0, 0] # Grab filter 0
            s = record                         # s is just 0-4 for the first half
            linear1.bias.data[:record] = cnv1.bias.data[0]

        # 4. Loop through the 3 elements of our chosen kernel
        for k in range(len(current_kernel)):
            c = s+k-pad # assigning a particular value of the kernel denoted by `k` at a certain place in weight matrix labelled as `s`
            # 5. Check if the column index is within the matrix bounds (0 to 4)
            if c>=0 and c<5:
                # Place the actual kernel weight into the matrix
                linear1.weight.data[record,c]=current_kernel[k]

print("Final Matrix:")
# print(linear1.weight.data)
# print(linear1.bias.data)

# 5. Feed both the CN and LIN layers with randomly generated tensor vector;
print(cnv1(random_vector))
print(linear1(random_vector))
message_21c = "This toy experiment yields results that could be easily perceived as hard proof and based upon which one can make the following case: A convolutional neural network is nothing else but a far smarter and well-constrained solution derived from the same basis as the one fully connected layers draw upon. The point of difference between FCN and a 1D-convolutional layer is entirely made from a dimensional mismatch at an output level: the output layer holds 10 neurons and activation maps (2, 5). In spite of this, the content remains exactly the same."
write_and_display_Message(message_21c, "2.1.C) Answer")

Final Matrix:
tensor([[[-0.5764,  0.9231,  1.9028,  0.0239,  0.4701],
         [-3.1388, -4.0631, -2.2972, -3.8415, -2.3503]]],
       grad_fn=<ConvolutionBackward0>)
tensor([[[-0.5764,  0.9231,  1.9028,  0.0239,  0.3920, -3.1388, -4.0631,
          -2.2972, -3.8415, -2.3503]]], grad_fn=<ViewBackward0>)


True

### 2.1.d) 

#### Toy CNN-Model

In [2]:
class DefaultCNN(nn.Module):
    def __init__(self, depth_no_cn_lay:int, width_no_kern:list, kernel_size:int|tuple=3):
        super().__init__()
        self.depth = depth_no_cn_lay
        self.width = width_no_kern
    
        # 0. Init super module that will later store all the sub-modules:|
        self.n = nn.Sequential()#______________________________________________________________
        # 1. Loop through the list holding the number of x-D convolutional layers to be created|
        for layer in range(self.depth):#________________________________________________
            # 2. Do explicit assignment of `ch_in, ch_out and pool type` for each layers|_____________________________________________
            self.n.append((nn.Conv1d(in_channels=width_no_kern[layer], out_channels=width_no_kern[layer+1], kernel_size=kernel_size)))#|
            self.n.append(nn.BatchNorm1d(width_no_kern[layer+1]))#
            self.n.append(nn.ReLU())#


    def train(self, sum:bool = False):
        if sum:
            print(summary(self.n), self.width)
        pass

In [3]:
model = DefaultCNN(depth_no_cn_lay=3, width_no_kern=[3, 32, 64, 32])
model.train(True)

Layer (type:depth-idx)                   Param #
Sequential                               --
├─Conv1d: 1-1                            320
├─BatchNorm1d: 1-2                       64
├─ReLU: 1-3                              --
├─Conv1d: 1-4                            6,208
├─BatchNorm1d: 1-5                       128
├─ReLU: 1-6                              --
├─Conv1d: 1-7                            6,176
├─BatchNorm1d: 1-8                       64
├─ReLU: 1-9                              --
Total params: 12,960
Trainable params: 12,960
Non-trainable params: 0
================================================================= [3, 32, 64, 32]


#### Training 

In [ ]:
args = get_parser_Arguments()
train_loader, val_loader, test_loader = create_dataloaders(args.data_dir, args.batch_size)/s

# Utils (helpers or static methods)

In [ ]:
@staticmethod
def write_and_display_Message(message:str, title:str)-> None:
    ct.windll.user32.MessageBoxW(0, message, title, 1)

    return True

In [4]:
@staticmethod
def get_parser_Arguments():
    parser = argparse.ArgumentParser(description="Experiment controller")
    parser.add_argument("--data_dir", type=str, default=r"/DataSets", help="Path where data should be store")
    parser.add_argument("--batch_size", type=int, default=64, help="Mini-batch size controller")

    return parser.parse_known_args()[0]

args = get_parser_Arguments()
print(args.batch_size)

64
